## algorithm design and anlysis-2026 spring  homework 2
**Deadline**：2026.5.20

**name**: 段飞 112025321342032


note：
---
1. 本题目为在线OJ作业，OJ平台题目链接：https://www.nowcoder.com/acm/contest/129481，访问密码见课件；
3. 在OJ平台运行通过后，将源码复制到本文件对应题目的下方代码框中；
4. 如若作答有雷同，全部取消成绩；


## A 排序

In [ ]:
import sys

sys.setrecursionlimit(1000000)


def main():
    data = list(map(int, sys.stdin.buffer.read().split()))
    if not data:
        return

    ptr = 0
    size_n = data[ptr]
    ptr += 1

    key_x = data[ptr]
    key_y = data[ptr + 1]
    ptr += 2

    cur = data[ptr:ptr + size_n]

    record = []

    def use_pair_swap():
        record.append(0)
        for i in range(size_n):
            if cur[i] == key_x:
                cur[i] = key_y
            elif cur[i] == key_y:
                cur[i] = key_x

    def use_add(delta):
        delta %= size_n
        if delta == 0:
            return
        record.append(delta)
        for i in range(size_n):
            cur[i] = (cur[i] + delta) % size_n

    def use_xor(mask):
        if mask == 0:
            return
        record.append(-mask)
        for i in range(size_n):
            cur[i] ^= mask

    class PatternBuilder:
        __slots__ = ("arr", "length", "ops")

        def __init__(self, arr):
            self.arr = arr
            self.length = len(arr)
            self.ops = []

        def build(self):
            m = self.length

            used = [False] * m
            for v in self.arr:
                if v < 0 or v >= m or used[v]:
                    return False
                used[v] = True

            if m == 1:
                return True

            half = m // 2

            left = PatternBuilder([self.arr[i * 2] // 2 for i in range(half)])
            right = PatternBuilder([self.arr[i * 2 + 1] // 2 for i in range(half)])

            if not left.build() or not right.build():
                return False

            seq = []

            if self.arr[0] & 1:
                seq.append(1 if m == 2 else -1)

            left_mask = 0
            for op in left.ops:
                if op > 0:
                    seq.append(-1)
                    seq.append(1)
                else:
                    val = op * 2
                    seq.append(val)
                    left_mask ^= -val

            if left_mask:
                seq.append(-left_mask)

            right_mask = 0
            for op in right.ops:
                if op > 0:
                    seq.append(1)
                    seq.append(-1)
                else:
                    val = op * 2
                    seq.append(val)
                    right_mask ^= -val

            if (right_mask & half) != (left_mask & half):
                return False

            if left_mask >= half:
                left_mask -= half
            if right_mask >= half:
                right_mask -= half

            if left_mask != right_mask:
                return False

            merged = []
            for op in seq:
                if merged and op < 0 and merged[-1] < 0:
                    new_mask = (-merged[-1]) ^ (-op)
                    if new_mask == 0:
                        merged.pop()
                    else:
                        merged[-1] = -new_mask
                else:
                    merged.append(op)

            self.ops = merged
            return True

    block = (key_x - key_y + size_n) % size_n
    block &= -block

    if block == 0:
        block = size_n

    if block > 1:
        base_arr = [cur[i] & (block - 1) for i in range(block)]
        builder = PatternBuilder(base_arr)

        if not builder.build():
            print(-1)
            return

        for op in builder.ops:
            if op > 0:
                use_add(op)
            else:
                use_xor(-op)

    def get_anchor(u, v):
        diff = (v - u + size_n - block) % size_n

        pos_u = 0
        pos_v = 0

        step = size_n // 2
        while step >= 2 * block:
            if diff >= step:
                diff -= step
                pos_v += step // 2
            else:
                pos_u += step // 2
            step //= 2

        low = u & (block - 1)
        pos_u += size_n // 2 + low
        pos_v += low

        return pos_u, pos_v

    def simulate_swap(u, v):
        if ((u // block) & 1) == ((v // block) & 1):
            if ((u // block) & 1) == 0:
                bridge = (u & (block - 1)) + block
            else:
                bridge = u & (block - 1)

            simulate_swap(u, bridge)
            simulate_swap(v, bridge)
            simulate_swap(u, bridge)
        else:
            base_pos, _ = get_anchor(key_x, key_y)
            target_pos, _ = get_anchor(u, v)

            use_add((target_pos - u) % size_n)
            use_xor(target_pos ^ base_pos)
            use_add((key_x - base_pos) % size_n)

            use_pair_swap()

            use_add((base_pos - key_x) % size_n)
            use_xor(target_pos ^ base_pos)
            use_add((u - target_pos) % size_n)

    for residue in range(block):
        values = []
        for idx in range(residue, size_n, block):
            values.append(cur[idx])

        values.sort()

        p = 0
        ok = True
        for idx in range(residue, size_n, block):
            if values[p] != idx:
                ok = False
                break
            p += 1

        if not ok:
            print(-1)
            return

        for idx in range(residue, size_n, block):
            if cur[idx] != idx:
                simulate_swap(idx, cur[idx])

    if cur != list(range(size_n)):
        print(-1)
        return

    out = [str(len(record))]

    for op in record:
        if op == 0:
            out.append("0")
        elif op < 0:
            out.append(f"1 {-op}")
        else:
            out.append(f"2 {op}")

    sys.stdout.write("\n".join(out))


if __name__ == "__main__":
    main()

## B 长跑

In [ ]:
## add your code here
import sys
import heapq


def solve_case(n, L, Maxn, S, stations):
    # 同一个位置可能有多个补给站，只保留最便宜的
    cost_map = {}

    for p, c in stations:
        if 0 <= p <= L:
            if p not in cost_map or c < cost_map[p]:
                cost_map[p] = c

    # 起点和终点加入关键点
    cost_map[0] = 0
    if L not in cost_map:
        cost_map[L] = 0

    positions = sorted(cost_map.keys())
    m = len(positions)

    # 如果起点就是终点
    if L == 0:
        return "Yes"

    INF = 10 ** 18
    dp = [INF] * m

    # 小根堆中存储：
    heap = []

    dp[0] = 0
    heapq.heappush(heap, (0, positions[0]))

    for i in range(1, m):
        cur_pos = positions[i]

        # 删除距离当前点超过Maxn的点
        while heap and cur_pos - heap[0][1] > Maxn:
            heapq.heappop(heap)

        # 没有任何点可以到达当前位置
        if not heap:
            continue

        # 当前点可以由堆顶最优点到达
        dp[i] = heap[0][0]

        # 如果当前位置是终点，不需要再补给
        if cur_pos == L:
            break

        # 如果花费已经超过S，可以不加入堆，后面只会更贵
        if dp[i] <= S:
            heapq.heappush(heap, (dp[i] + cost_map[cur_pos], cur_pos))

    end_index = positions.index(L)

    if dp[end_index] <= S:
        return "Yes"
    else:
        return "No"


def main():
    data = sys.stdin.buffer.read().split()
    idx = 0
    ans = []

    while idx < len(data):
        n = int(data[idx])
        L = int(data[idx + 1])
        Maxn = int(data[idx + 2])
        S = int(data[idx + 3])
        idx += 4

        stations = []
        for _ in range(n):
            p = int(data[idx])
            c = int(data[idx + 1])
            idx += 2
            stations.append((p, c))

        ans.append(solve_case(n, L, Maxn, S, stations))

    sys.stdout.write("\n".join(ans))


if __name__ == "__main__":
    main()

## C 最长回文

In [ ]:
## add your code here
import sys

MASK = (1 << 64) - 1
BASE = 13331


def build_transformed(s: bytes) -> bytes:
    arr = bytearray()
    arr.append(64)  # '@'
    for ch in s:
        arr.append(35)  # '#'
        arr.append(ch)
    arr.append(35)  # '#'
    return bytes(arr)


def manacher(t: bytes):
    n = len(t)
    p = [0] * n
    center = 0
    right = 0

    for i in range(1, n):
        if right > i:
            mirror = 2 * center - i
            p[i] = min(p[mirror], right - i)
        else:
            p[i] = 1

        while i - p[i] >= 0 and i + p[i] < n and t[i - p[i]] == t[i + p[i]]:
            p[i] += 1

        if i + p[i] > right:
            right = i + p[i]
            center = i

    return p


def solution(n, a, b):
    s1 = build_transformed(a)
    s2 = build_transformed(b)

    m = len(s1)

    len1 = manacher(s1)
    len2 = manacher(s2)

    rev1 = s1[::-1]

    power = [1] * (m + 1)
    for i in range(1, m + 1):
        power[i] = (power[i - 1] * BASE) & MASK

    h1 = [0] * (m + 1)
    h2 = [0] * (m + 1)

    for i in range(m):
        h1[i + 1] = (h1[i] * BASE + rev1[i]) & MASK
        h2[i + 1] = (h2[i] * BASE + s2[i]) & MASK

    ans = 0

    pwr = power
    hr = h1
    hs = h2
    mask = MASK

    for i in range(2, m):
        tmp = len1[i]
        v = len2[i - 2]
        if v > tmp:
            tmp = v

        p = i - tmp
        q = i - 2 + tmp

        cur = tmp - 1
        if cur > ans:
            ans = cur

        if p < 0 or q >= m:
            continue

        if s1[p] != s2[q]:
            continue

        max_ext = p + 1
        right_ext = m - q
        if right_ext < max_ext:
            max_ext = right_ext


        if tmp - 1 + max_ext <= ans:
            continue

        start1 = m - 1 - p
        start2 = q

        def same(length):
            x1 = start1
            y1 = start1 + length
            x2 = start2
            y2 = start2 + length

            hash_a = (hr[y1] - ((hr[x1] * pwr[length]) & mask)) & mask
            hash_b = (hs[y2] - ((hs[x2] * pwr[length]) & mask)) & mask

            return hash_a == hash_b

        low = 1
        step = 2

        while step <= max_ext and same(step):
            low = step
            step <<= 1

        left = low
        right = min(step - 1, max_ext)

        while left < right:
            mid = (left + right + 1) >> 1
            if same(mid):
                left = mid
            else:
                right = mid - 1

        total = tmp + left - 1
        if total > ans:
            ans = total

    return ans


def main():
    data = sys.stdin.buffer.read().split()
    n = int(data[0])
    a = data[1]
    b = data[2]

    print(solution(n, a, b))


if __name__ == "__main__":
    main()

## D 优惠券

In [ ]:
## add your code here
import sys

MAXX = 100000 + 5

class BIT:
    def __init__(self, n):
        self.n = n
        self.tree = [0] * (n + 2)
        self.total = 0

    def add(self, i, v):
        self.total += v
        n = self.n
        tree = self.tree
        while i <= n:
            tree[i] += v
            i += i & -i

    def sum(self, i):
        if i <= 0:
            return 0
        if i > self.n:
            i = self.n

        res = 0
        tree = self.tree
        while i > 0:
            res += tree[i]
            i -= i & -i
        return res

    def kth(self, k):
        #返回最小 pos，使得 prefix_sum(pos) >= k
        pos = 0
        bit = 1
        while (bit << 1) <= self.n:
            bit <<= 1

        tree = self.tree
        while bit:
            nxt = pos + bit
            if nxt <= self.n and tree[nxt] < k:
                pos = nxt
                k -= tree[nxt]
            bit >>= 1

        return pos + 1

    def erase_first_greater_than(self, x):
        # 删除最小的、位置 > x 的问号。
        # 如果不存在，返回 False。

        cnt_le = self.sum(x)

        if cnt_le >= self.total:
            return False

        pos = self.kth(cnt_le + 1)
        self.add(pos, -1)
        return True


def solve_case(m, stdin):
    bit = BIT(m)

    # a[x]：当前 x 最后一次未被匹配掉的 I 的位置
    # d[x]：当前 x 最后一次 O 的位置
    a = [0] * MAXX
    d = [0] * MAXX

    ans = -1

    for i in range(1, m + 1):
        line = stdin.readline()

        while line and line.strip() == b"":
            line = stdin.readline()

        parts = line.split()
        op = parts[0]

        if op == b"?" or op == "？".encode():
            if ans == -1:
                bit.add(i, 1)
            continue

        x = int(parts[1])

        if ans != -1:
            continue

        if op == b"I":
            # 如果之前已经有未使用的 I x，
            # 那么中间必须有一个 ? 
            if a[x] != 0:
                if not bit.erase_first_greater_than(a[x]):
                    ans = i

            a[x] = i

        else:  # op == b"O"，大写字母 O
            # 如果当前没有未使用的 I x，
            # 那么必须有一个 ? 
            if a[x] == 0:
                if not bit.erase_first_greater_than(d[x]):
                    ans = i

            a[x] = 0
            d[x] = i

    return ans


def main():
    stdin = sys.stdin.buffer
    ans = []

    while True:
        line = stdin.readline()

        if not line:
            break

        line = line.strip()

        if not line:
            continue

        m = int(line)
        ans.append(str(solve_case(m, stdin)))

    sys.stdout.write("\n".join(ans))


if __name__ == "__main__":
    main()

## E 任意点

In [ ]:
## add your code here
import sys

def solution(n):
    points = []
    index = 1
    for _ in range(n):
        x = int(input_data[index])
        y = int(input_data[index+1])
        points.append((x, y))
        index += 2
        

    parent = list(range(n))
    
    # 查找根节点
    def find(i):
        if parent[i] != i:
            parent[i] = find(parent[i])
        return parent[i]
        
    # 合并两个集合
    def union(i, j):
        root_i = find(i)
        root_j = find(j)
        if root_i != root_j:
            parent[root_i] = root_j
            
    # 遍历所有点对，如果x同或y同，则合并连通块
    for i in range(n):
        for j in range(i + 1, n):
            if points[i][0] == points[j][0] or points[i][1] == points[j][1]:
                union(i, j)
                
    #  统计最终有多少个独立的连通块
    roots = set()
    for i in range(n):
        roots.add(find(i))
        
    # 至少需要的点数=连通块数量-1
    res = len(roots) - 1
    print(res)

if __name__ == '__main__':
    input_data = sys.stdin.read().split()
    n = int(input_data[0])
    solution(n)

## F 通配符匹配

In [ ]:
import sys

STAR = 42  
QUES = 63   

class Segment:
    __slots__ = ("raw", "length", "pieces", "all_question", "exact")

    def __init__(self, raw: bytes):
        self.raw = raw
        self.length = len(raw)

        pieces = []
        i = 0
        n = len(raw)

        while i < n:
            if raw[i] == QUES:
                i += 1
                continue

            j = i
            while j < n and raw[j] != QUES:
                j += 1

            pieces.append((i, raw[i:j]))
            i = j

        self.pieces = pieces
        self.all_question = len(pieces) == 0
        self.exact = len(pieces) == 1 and pieces[0][0] == 0 and len(pieces[0][1]) == n

    def match_at(self, text: bytes, pos: int) -> bool:
        if pos < 0 or pos + self.length > len(text):
            return False

        if self.length == 0:
            return True

        if self.all_question:
            return True

        if self.exact:
            return text.startswith(self.raw, pos)

        for off, lit in self.pieces:
            if not text.startswith(lit, pos + off):
                return False

        return True

    def find_earliest(self, text: bytes, left: int, right: int) -> int:

        # 在 text[left:right] 中找最早的起点 pos，
        # 使得该 Segment 可以完整匹配 text[pos:pos+length]。
        # 找不到返回 -1。

        L = self.length

        if L == 0:
            return left

        max_start = right - L
        if left > max_start:
            return -1

        if self.all_question:
            return left

        if self.exact:
            pos = text.find(self.raw, left, right)
            if pos == -1 or pos > max_start:
                return -1
            return pos

        # 选择当前搜索区间里出现次数最少的普通片段作为锚点
        best_off = -1
        best_lit = b""
        best_cnt = None
        best_len = -1

        for off, lit in self.pieces:
            start = left + off
            end = max_start + off + len(lit)

            cnt = text.count(lit, start, end)

            if best_cnt is None or cnt < best_cnt or (cnt == best_cnt and len(lit) > best_len):
                best_cnt = cnt
                best_off = off
                best_lit = lit
                best_len = len(lit)

                if cnt == 0:
                    return -1

        search_start = left + best_off
        search_end = max_start + best_off + len(best_lit)

        pos = text.find(best_lit, search_start, search_end)

        while pos != -1:
            start_pos = pos - best_off

            ok = True
            for off, lit in self.pieces:
                if not text.startswith(lit, start_pos + off):
                    ok = False
                    break

            if ok:
                return start_pos

            pos = text.find(best_lit, pos + 1, search_end)

        return -1


def compress_stars(pattern: bytes) -> bytes:
    res = bytearray()
    last_star = False

    for ch in pattern:
        if ch == STAR:
            if not last_star:
                res.append(ch)
                last_star = True
        else:
            res.append(ch)
            last_star = False

    return bytes(res)


def build(pattern: bytes):
    pattern = compress_stars(pattern)
    has_star = STAR in pattern
    min_len = sum(1 for ch in pattern if ch != STAR)

    if not has_star:
        return has_star, min_len, [Segment(pattern)]

    parts = pattern.split(b"*")
    segs = [Segment(part) for part in parts]

    return has_star, min_len, segs


def match_no_star(seg: Segment, text: bytes) -> bool:
    return seg.length == len(text) and seg.match_at(text, 0)


def match_with_star(segs, min_len: int, text: bytes) -> bool:
    n = len(text)

    if n < min_len:
        return False

    left = 0
    right = n

    # 第一个片段必须贴着开头匹配
    prefix = segs[0]
    if prefix.length:
        if not prefix.match_at(text, 0):
            return False
        left = prefix.length

    # 最后一个片段必须贴着结尾匹配
    suffix = segs[-1]
    if suffix.length:
        start = n - suffix.length
        if start < left:
            return False
        if not suffix.match_at(text, start):
            return False
        right = start

    # 中间片段按顺序出现即可
    for seg in segs[1:-1]:
        if seg.length == 0:
            continue

        pos = seg.find_earliest(text, left, right)

        if pos == -1:
            return False

        left = pos + seg.length

        if left > right:
            return False

    return True


def main():
    data = sys.stdin.buffer.read().split()

    pattern = data[0]
    q = int(data[1])

    has_star, min_len, segs = build(pattern)

    ans = []
    idx = 2

    if not has_star:
        seg = segs[0]
        for _ in range(q):
            text = data[idx]
            idx += 1

            ans.append("YES" if match_no_star(seg, text) else "NO")
    else:
        for _ in range(q):
            text = data[idx]
            idx += 1

            ans.append("YES" if match_with_star(segs, min_len, text) else "NO")

    sys.stdout.write("\n".join(ans))


if __name__ == "__main__":
    main()

## G 汉诺塔

In [ ]:
## add your code here
import sys

def HanoiB():
    input_data = sys.stdin.read().split()
    if not input_data:
        return
    
    n = int(input_data[0])
    pri_list = input_data[1:7]
    
    #值越小优先级越高
    pri = {move: index for index, move in enumerate(pri_list)}
    #三根柱子
    pegs = ['A', 'B', 'C']
    
    # dp[i][u] 表示从柱子u移走i个盘子所需的步数
    # dest[i][u] 表示从柱子u移走i盘子最终的落点
    dp = {i: {peg: 0 for peg in pegs} for i in range(1, n + 1)}
    dest = {i: {peg: '' for peg in pegs} for i in range(1, n + 1)}
    
    # i=1
    for u in pegs:
        # 找出另外两个柱子
        other_pegs = [p for p in pegs if p != u]
        v1, v2 = other_pegs[0], other_pegs[1]
        
        # 比较操作的优先级
        move1 = u + v1
        move2 = u + v2
        
        if pri[move1] < pri[move2]:
            preferred_target = v1
        else:
            preferred_target = v2
            
        dp[1][u] = 1
        dest[1][u] = preferred_target

    # 状态转移 (i从2到n)
    for i in range(2, n + 1):
        for u in pegs:
            # i-1个盘子自然移动到w
            w = dest[i-1][u]
            cost1 = dp[i-1][u]
            
            #第i号盘子移动到空闲的第三根柱子v
            v = [p for p in pegs if p not in (u, w)][0]
            
            #i-1个盘子从w继续自然移动
            next_dest = dest[i-1][w]
            cost2 = dp[i-1][w]
            
            if next_dest == v:
                #在v柱汇合
                dp[i][u] = cost1 + 1 + cost2
                dest[i][u] = v
            else:
                #i-1个盘子回到了u，需要额外搬运过程才能在w汇合
                dp[i][u] = 2 * cost1 + cost2 + 2
                dest[i][u] = w

    #从A柱子移动n个盘子的总步数
    print(dp[n]['A'])

if __name__ == '__main__':
    HanoiB()

## H 马步距离

In [ ]:
## add your code here
import sys

def get_min_moves(xp, yp, xs, ys):
    #计算相对绝对距离
    dx = abs(xp - xs)
    dy = abs(yp - ys)
    
    #确保dx >= dy
    if dx < dy:
        dx, dy = dy, dx

    if dx == 1 and dy == 0:
        return 3
    if dx == 2 and dy == 2:
        return 4
        
    #计算下界
    m1 = (dx + 1) // 2
    m2 = (dx + dy + 2) // 3
    mvs = max(m1, m2)
    
    # 奇偶性校验
    if (mvs % 2) != ((dx + dy) % 2):
        mvs += 1
        
    return mvs

if __name__ == "__main__":
    input_data = sys.stdin.read().split()
    if input_data:
        xp, yp, xs, ys = map(int, input_data[:4])
        print(get_min_moves(xp, yp, xs, ys))

## I 直方图最大矩形

In [ ]:
## add your code here
class Solution:
    def largestRectangleArea(self, heights):
        # 边界条件：如果输入为空，直接返回0
        if not heights:
            return 0
            
        # 在末尾添加一个高度为0的元素，确保最后能把栈里的元素全部出栈计算
        heights.append(0)
        stack = []  # 保存下标
        max_area = 0
        
        for i in range(len(heights)):
            # 当前柱子高度小于栈顶柱子高度，找到栈顶元素的右边界
            while stack and heights[i] < heights[stack[-1]]:
                # 弹出栈顶元素, 矩形高度
                h = heights[stack.pop()]
                
                # 矩形的宽度
                if not stack:
                    # 栈为空，出栈的柱子是最矮的，一直到最左侧
                    w = i 
                else:
                    # 栈不为空，左边界是新的栈顶元素，右边界是当前的i
                    w = i - stack[-1] - 1
                
                # 更新最大面积
                max_area = max(max_area, h * w)
            
            # 当前元素的下标入栈
            stack.append(i)
            
        return max_area

## J 消防局的设立

In [ ]:
## add your code here
import sys

def solve(n):
    if n == 0:
        print(0)
        return
        
    #建立邻接表，下标从1到n
    adj = [[] for _ in range(n + 1)]
    
    #读取n-1条边，第i行表示从节点i到a[i]的边
    for i in range(1, n):
        u = i + 1
        v = int(input_data[i])
        adj[u].append(v)
        adj[v].append(u)
        
    parent = [0] * (n + 1)
    
    #BFS找每个节点的父节点，并按层级顺序记录节点
    order = [1]
    visited = [False] * (n + 1)
    visited[1] = True
    
    index = 0
    while index < len(order):
        curr = order[index]
        index += 1
        for nt in adj[curr]:
            if not visited[nt]:
                visited[nt] = True
                parent[nt] = curr
                order.append(nt)
                
    covered = [False] * (n + 1)
    ans = 0
    
    #逆序遍历order，按深度从深到浅处理节点
    for i in range(n - 1, -1, -1):
        u = order[i]
        
        #若节点已经被覆盖，则跳过
        if covered[u]:
            continue
            
        #寻找建消防局的最佳位置
        p = u
        if parent[p] != 0:
            p = parent[p]
        if parent[p] != 0:
            p = parent[p]
            
        #在p处建立消防局
        ans += 1
        
        # 标记p及距离p不超过2的所有节点为已覆盖
        covered[p] = True
        for v1 in adj[p]:
            covered[v1] = True
            for v2 in adj[v1]:
                covered[v2] = True
                
    #最小消防局数量
    return ans

if __name__ == '__main__':
    input_data = sys.stdin.read().split() #读取数据
    n = int(input_data[0])
    res = solve(n)
    print(res)